In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores four performance characteristics of a
# continuous-time elliptic (Cauer) low-pass filter:
#
#       1. Group delay τg(ω)
#       2. Loss characteristic A(ω)
#       3. Selectivity Fs
#       4. Spectral performance factor Sαβ
#
# Six parameters can be varied:
#
#       N   : filter order
#       ωp  : passband-edge angular frequency
#       Ap  : maximum passband loss / ripple in dB
#       As  : minimum stopband attenuation in dB
#       α   : lower attenuation level used for Sαβ
#       β   : higher attenuation level used for Sαβ
#
# The analog low-pass filter is constructed directly with
#
#       scipy.signal.ellip(..., analog=True)
#
# Group delay:
#
#       τg(ω) = -dφ(ω)/dω
#
# Loss characteristic:
#
#       A(ω) = -20 log10 |H(jω)|
#
# Selectivity is evaluated numerically from
#
#       Fs = -d|H(jω)|/dω
#
# at the first frequency where the attenuation reaches 3.0103 dB.
#
# Spectral performance:
#
#       Sαβ = ωβ / ωα
#
# where ωα and ωβ are the first frequencies at which the attenuation reaches
# α and β dB respectively.
#
# The attenuation levels α and β affect only the spectral performance factor.
#
# Elliptic filters exhibit equiripple behavior in BOTH the passband and
# stopband and contain finite transmission zeros in the stopband.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:820px;
    max-width:820px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the group delay, loss characteristic, selectivity and spectral performance factor of continuous-time elliptic low-pass filters.
<br>
<b>Interpretation:</b>
The sliders control N, ω<sub>p</sub>, passband ripple A<sub>p</sub> and
stopband attenuation A<sub>s</sub>. Elliptic filters exhibit equal-ripple
behavior in both passband and stopband and achieve an exceptionally sharp
transition. The attenuation levels α and β are used only for the spectral
performance factor S<sub>α</sub><sup>β</sup>.
</div>
""", layout=Layout(width='830px', max_width='830px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=2, max=10, step=1, value=4, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

wp_slider = FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description='ωp:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

Ap_slider = FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='Ap (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

As_slider = FloatSlider(min=30.0, max=80.0, step=1.0, value=50.0, description='As (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

alpha_slider = FloatSlider(min=3.0, max=15.0, step=0.5, value=7.0, description='α (dB):', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

beta_slider = FloatSlider(min=20.0, max=25.0, step=1.0, value=20.0, description='β (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='290px', max_width='290px'))

# ==============================================================================
# FIGURE 1: GROUP DELAY
# ==============================================================================

fig_gd, ax_gd = plt.subplots(figsize=(4.6, 3.0))

gd_line, = ax_gd.plot([], [], 'r-', linewidth=2.0, label='τg(ω)')
wp_gd_line = ax_gd.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωp')
zero_gd_line = ax_gd.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_gd.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_gd.set_ylabel('Group Delay τg(ω) (s)', fontsize=9)
ax_gd.set_title('Group Delay', fontsize=11, fontweight='bold', pad=5)
ax_gd.tick_params(axis='both', labelsize=8)
ax_gd.grid(True, linestyle=':', alpha=0.5)
ax_gd.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)

ax_gd.set_xlim(0.0, 10.0)
ax_gd.set_ylim(0.0, 80.0)
ax_gd.set_yticks(np.arange(0.0, 81.0, 10.0))

fig_gd.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_gd.canvas.header_visible = False
fig_gd.canvas.toolbar_visible = False
fig_gd.canvas.resizable = False
fig_gd.canvas.layout.width = '460px'
fig_gd.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 2: LOSS CHARACTERISTIC
# ==============================================================================

fig_loss, ax_loss = plt.subplots(figsize=(4.6, 3.0))

loss_line, = ax_loss.plot([], [], 'r-', linewidth=2.0, label='A(ω)')
wp_loss_line = ax_loss.axvline(1.0, color='black', linestyle=':', linewidth=1.1, label='ωp')
Ap_loss_line = ax_loss.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='Ap')
As_loss_line = ax_loss.axhline(50.0, color='black', linestyle='--', linewidth=0.9, label='As')

ax_loss.set_xscale('log')
ax_loss.set_xlabel('Angular Frequency ω (rad/s)', fontsize=9)
ax_loss.set_ylabel('Loss A(ω) (dB)', fontsize=9)
ax_loss.set_title('Loss Characteristic', fontsize=11, fontweight='bold', pad=5)
ax_loss.tick_params(axis='both', labelsize=8)
ax_loss.grid(True, which='both', linestyle=':', alpha=0.5)
ax_loss.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=4, fontsize=7)

ax_loss.set_xlim(1e-2, 20.0)
ax_loss.set_ylim(0.0, 110.0)

fig_loss.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_loss.canvas.header_visible = False
fig_loss.canvas.toolbar_visible = False
fig_loss.canvas.resizable = False
fig_loss.canvas.layout.width = '460px'
fig_loss.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 3: SELECTIVITY
# ==============================================================================

fig_sel, ax_sel = plt.subplots(figsize=(4.6, 3.0))

sel_line, = ax_sel.plot([], [], 'r-', linewidth=2.0, label='Fs(N)')
sel_point, = ax_sel.plot([], [], 'ro', markersize=6, label='Current N')

ax_sel.set_xlabel('Filter Order N', fontsize=9)
ax_sel.set_ylabel('Selectivity Fs', fontsize=9)
ax_sel.set_title('Selectivity', fontsize=11, fontweight='bold', pad=5)
ax_sel.tick_params(axis='both', labelsize=8)
ax_sel.grid(True, linestyle=':', alpha=0.5)
ax_sel.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=7)

ax_sel.set_xlim(2.0, 10.0)
ax_sel.set_xticks(np.arange(2, 11, 1))
ax_sel.set_ylim(0.0, 80.0)

fig_sel.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sel.canvas.header_visible = False
fig_sel.canvas.toolbar_visible = False
fig_sel.canvas.resizable = False
fig_sel.canvas.layout.width = '460px'
fig_sel.canvas.layout.height = '305px'

# ==============================================================================
# FIGURE 4: SPECTRAL PERFORMANCE FACTOR
# ==============================================================================

fig_sp, ax_sp = plt.subplots(figsize=(4.6, 3.0))

sp_line, = ax_sp.plot([], [], 'r-', linewidth=2.0, label='Sαβ(N)')
sp_point, = ax_sp.plot([], [], 'ro', markersize=6, label='Current N')
ideal_line = ax_sp.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='Ideal = 1')

ax_sp.set_xlabel('Filter Order N', fontsize=9)
ax_sp.set_ylabel('Spectral Factor Sαβ', fontsize=9)
ax_sp.set_title('Spectral Performance Factor', fontsize=11, fontweight='bold', pad=5)
ax_sp.tick_params(axis='both', labelsize=8)
ax_sp.grid(True, linestyle=':', alpha=0.5)
ax_sp.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=3, fontsize=7)

ax_sp.set_xlim(2.0, 10.0)
ax_sp.set_xticks(np.arange(2, 11, 1))
ax_sp.set_ylim(1.0, 10.0)

fig_sp.subplots_adjust(left=0.14, right=0.97, bottom=0.31, top=0.85)
fig_sp.canvas.header_visible = False
fig_sp.canvas.toolbar_visible = False
fig_sp.canvas.resizable = False
fig_sp.canvas.layout.width = '460px'
fig_sp.canvas.layout.height = '305px'

# ==============================================================================
# FREQUENCY AXES
# ==============================================================================

omega_gd = np.linspace(0.001, 10.0, 6000)
omega_loss = np.logspace(-2, np.log10(20.0), 8000)
omega_dense = np.logspace(-4, 3, 30000)
N_values = np.arange(2, 11)

# ==============================================================================
# AUXILIARY FUNCTIONS
# ==============================================================================

def elliptic_system(N, wp, Ap, As):

    z, p, k = signal.ellip(N, Ap, As, wp, btype='low', analog=True, output='zpk')

    b, a = signal.zpk2tf(z, p, k)

    return b, a

def first_attenuation_frequency(b, a, level_db):

    _, H = signal.freqs(b, a, worN=omega_dense)

    loss = -20.0 * np.log10(np.maximum(np.abs(H), 1e-15))

    indices = np.where(loss >= level_db)[0]

    if len(indices) == 0:
        return np.nan

    i = indices[0]

    if i == 0:
        return omega_dense[0]

    w1 = omega_dense[i - 1]
    w2 = omega_dense[i]

    A1 = loss[i - 1]
    A2 = loss[i]

    if A2 == A1:
        return w2

    w_level = w1 + (level_db - A1) * (w2 - w1) / (A2 - A1)

    return w_level

def elliptic_selectivity(N, wp, Ap, As):

    b, a = elliptic_system(N, wp, Ap, As)

    _, H = signal.freqs(b, a, worN=omega_dense)

    magnitude = np.abs(H)

    loss = -20.0 * np.log10(np.maximum(magnitude, 1e-15))

    indices = np.where(loss >= 3.01029995664)[0]

    if len(indices) == 0:
        return np.nan, np.nan

    i = indices[0]

    if i < 2 or i >= len(omega_dense) - 2:
        return np.nan, omega_dense[i]

    wc = omega_dense[i]

    dmag_dw = np.gradient(magnitude, omega_dense)

    Fs = -dmag_dw[i]

    return Fs, wc

def elliptic_spectral_factor(N, wp, Ap, As, alpha, beta):

    b, a = elliptic_system(N, wp, Ap, As)

    wa = first_attenuation_frequency(b, a, alpha)

    wb = first_attenuation_frequency(b, a, beta)

    if np.isnan(wa) or np.isnan(wb) or wa <= 0.0:
        return np.nan, wa, wb

    S = wb / wa

    return S, wa, wb

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_elliptic_performance(change=None):

    N = order_slider.value
    wp = wp_slider.value
    Ap = Ap_slider.value
    As = As_slider.value
    alpha = alpha_slider.value
    beta = beta_slider.value

    # --------------------------------------------------------------------------
    # ELLIPTIC TRANSFER FUNCTION
    # --------------------------------------------------------------------------

    b, a = elliptic_system(N, wp, Ap, As)

    # --------------------------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------------------------

    _, H_gd = signal.freqs(b, a, worN=omega_gd)

    phase = np.unwrap(np.angle(H_gd))

    group_delay = -np.gradient(phase, omega_gd)

    group_delay = np.where(group_delay >= 0.0, group_delay, np.nan)

    # --------------------------------------------------------------------------
    # LOSS CHARACTERISTIC
    # --------------------------------------------------------------------------

    _, H_loss = signal.freqs(b, a, worN=omega_loss)

    magnitude_loss = np.maximum(np.abs(H_loss), 1e-15)

    loss = -20.0 * np.log10(magnitude_loss)

    # --------------------------------------------------------------------------
    # SELECTIVITY
    # --------------------------------------------------------------------------

    Fs, wc = elliptic_selectivity(N, wp, Ap, As)

    Fs_values = np.array([elliptic_selectivity(n, wp, Ap, As)[0] for n in N_values])

    # --------------------------------------------------------------------------
    # SPECTRAL PERFORMANCE FACTOR
    # --------------------------------------------------------------------------

    S, BW_alpha, BW_beta = elliptic_spectral_factor(N, wp, Ap, As, alpha, beta)

    S_values = np.array([elliptic_spectral_factor(n, wp, Ap, As, alpha, beta)[0] for n in N_values])

    # --------------------------------------------------------------------------
    # GROUP DELAY UPDATE
    # --------------------------------------------------------------------------

    gd_line.set_data(omega_gd, group_delay)

    wp_gd_line.set_xdata([wp, wp])

    ax_gd.set_xlim(0.0, 10.0)

    ax_gd.set_ylim(0.0, 80.0)

    # --------------------------------------------------------------------------
    # LOSS CHARACTERISTIC UPDATE
    # --------------------------------------------------------------------------

    loss_line.set_data(omega_loss, loss)

    wp_loss_line.set_xdata([wp, wp])

    Ap_loss_line.set_ydata([Ap, Ap])

    As_loss_line.set_ydata([As, As])

    ax_loss.set_xlim(1e-2, 20.0)

    ax_loss.set_ylim(0.0, 110.0)

    # --------------------------------------------------------------------------
    # SELECTIVITY UPDATE
    # --------------------------------------------------------------------------

    sel_line.set_data(N_values, Fs_values)

    if np.isfinite(Fs):
        sel_point.set_data([N], [Fs])
    else:
        sel_point.set_data([], [])

    ax_sel.set_xlim(2.0, 10.0)

    ax_sel.set_ylim(0.0, 80.0)

    # --------------------------------------------------------------------------
    # SPECTRAL PERFORMANCE FACTOR UPDATE
    # --------------------------------------------------------------------------

    sp_line.set_data(N_values, S_values)

    if np.isfinite(S):
        sp_point.set_data([N], [S])
    else:
        sp_point.set_data([], [])

    ax_sp.set_xlim(2.0, 10.0)

    ax_sp.set_ylim(1.0, 10.0)

    # --------------------------------------------------------------------------
    # INFORMATION PANEL
    # --------------------------------------------------------------------------

    Fs_text = f'{Fs:.6f}' if np.isfinite(Fs) else 'undefined'
    wc_text = f'{wc:.6f} rad/s' if np.isfinite(wc) else 'undefined'
    BW_alpha_text = f'{BW_alpha:.6f} rad/s' if np.isfinite(BW_alpha) else 'undefined'
    BW_beta_text = f'{BW_beta:.6f} rad/s' if np.isfinite(BW_beta) else 'undefined'
    S_text = f'{S:.6f}' if np.isfinite(S) else 'undefined'

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.55;
        background:white;
        width:285px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Elliptic (Cauer) low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Passband edge:</b>
        <span style="color:#0066cc;">ωp = {wp:.2f} rad/s</span>
    </div>

    <div>
        <b>Passband ripple:</b>
        <span style="color:#0066cc;">Ap = {Ap:.2f} dB</span>
    </div>

    <div>
        <b>Stopband attenuation:</b>
        <span style="color:#0066cc;">As = {As:.1f} dB</span>
    </div>

    <div>
        <b>3-dB cutoff:</b>
        <span style="color:#0066cc;">ωc = {wc_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Performance quantities:</b>
    </div>

    <div>
        <b>Selectivity:</b>
        <span style="color:#0066cc;">Fs = {Fs_text}</span>
    </div>

    <div>
        <b>Attenuation levels:</b>
        <span style="color:#0066cc;">α = {alpha:.1f} dB, β = {beta:.1f} dB</span>
    </div>

    <div>
        <b>BWα:</b>
        <span style="color:#0066cc;">{BW_alpha_text}</span>
    </div>

    <div>
        <b>BWβ:</b>
        <span style="color:#0066cc;">{BW_beta_text}</span>
    </div>

    <div>
        <b>Spectral factor:</b>
        <span style="color:#0066cc;">Sαβ = {S_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Elliptic filters exhibit equal-ripple behavior in both passband and
        stopband and provide the sharpest transition among the classical
        approximations for comparable specifications. Transmission zeros
        produce theoretically infinite loss at discrete stopband frequencies.
        The group delay is strongly frequency dependent near the transition.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_gd.canvas.draw_idle()
    fig_loss.canvas.draw_idle()
    fig_sel.canvas.draw_idle()
    fig_sp.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_elliptic_performance, names='value')
wp_slider.observe(update_elliptic_performance, names='value')
Ap_slider.observe(update_elliptic_performance, names='value')
As_slider.observe(update_elliptic_performance, names='value')
alpha_slider.observe(update_elliptic_performance, names='value')
beta_slider.observe(update_elliptic_performance, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, wp_slider, Ap_slider, As_slider, alpha_slider, beta_slider, info_html], layout=Layout(width='300px', min_width='300px', max_width='300px', flex='0 0 300px', align_items='flex-start'))

top_row = HBox([fig_gd.canvas, fig_loss.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_sel.canvas, fig_sp.canvas], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='930px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1230px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_elliptic_performance()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_layout)